# Rust Harness Adapter Test

This notebook validates that the **Rust `claw` binary** works end-to-end with a
**local Ollama model**, and can act as a manager agent by calling the
**Python autoresearch REST API server**.

Test sequence:

1. Verify prerequisites (cargo, Ollama reachable)
2. Build `claw` from source (skipped if the binary already exists)
3. Confirm `claw --help` runs cleanly
4. Ping Ollama through `claw` (one-shot prompt via OpenAI-compat path)
5. `claw --output-format json` produces parseable JSON
6. Start the Python API server (`api-server` subcommand)
7. `curl` the API server's `/health` and `/status` endpoints directly
8. Ask `claw` (via Ollama) to call the API server using its bash tool
9. Verify the round-trip: Rust LLM → API → Python control plane


In [1]:
from __future__ import annotations

import json
import os
import shlex
import signal
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path


def find_claw_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "main.py").exists() and (candidate / "tests" / "test_porting_workspace.py").exists():
            return candidate
    raise RuntimeError(f"Could not locate claw-code root from {current}")


CLAW_ROOT = find_claw_root()
RUST_DIR = CLAW_ROOT / "rust"
CLAW_BIN = RUST_DIR / "target" / "debug" / "claw"
WORKSPACE_ROOT = CLAW_ROOT.parent.parent
AUTORESEARCH_ROOT = WORKSPACE_ROOT / "nodes" / "autoresearch-macos"
PYTHON = sys.executable

# ── helper: run a subprocess and print output ─────────────────────────────────
def run_cmd(
    args: list[str],
    cwd: Path | None = None,
    env: dict | None = None,
    check: bool = True,
    timeout: int = 300,
) -> subprocess.CompletedProcess:
    effective_env = {**os.environ, **(env or {})}
    print("$", " ".join(shlex.quote(str(a)) for a in args))
    result = subprocess.run(
        [str(a) for a in args],
        cwd=str(cwd or CLAW_ROOT),
        stdin=subprocess.DEVNULL,   # prevent read_piped_stdin() blocking on Jupyter's open stdin
        capture_output=True,
        text=True,
        timeout=timeout,
        env=effective_env,
    )
    if result.stdout.strip():
        print(result.stdout[:2000])
    if result.stderr.strip():
        print(result.stderr[:1000], file=sys.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(
            f"command failed (exit {result.returncode}): "
            + (result.stderr.strip() or result.stdout.strip() or "no output")[:400]
        )
    return result


def run_json(args: list[str], cwd: Path | None = None, env: dict | None = None, check: bool = True):
    result = run_cmd(args, cwd=cwd, env=env, check=check)
    return json.loads(result.stdout)


def ollama_ready(host: str = "http://localhost:11434") -> bool:
    try:
        with urllib.request.urlopen(f"{host.rstrip('/')}/api/tags", timeout=3) as r:
            return r.status == 200
    except (urllib.error.URLError, TimeoutError, OSError):
        return False


def http_get_json(url: str, timeout: int = 5) -> dict:
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return json.loads(r.read())


def wait_for_json(url: str, timeout: float = 30.0, interval: float = 0.5) -> dict:
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            return http_get_json(url, timeout=3)
        except Exception as e:
            last_error = e
            time.sleep(interval)
    raise RuntimeError(f"Timed out waiting for {url}: {last_error}")


def latest_rust_source_mtime(root: Path) -> float:
    patterns = ("*.rs", "Cargo.toml", "Cargo.lock")
    latest = 0.0
    for pattern in patterns:
        for path in root.rglob(pattern):
            if "/target/" in path.as_posix():
                continue
            latest = max(latest, path.stat().st_mtime)
    return latest


print({
    "claw_root": str(CLAW_ROOT),
    "rust_dir": str(RUST_DIR),
    "autoresearch_root": str(AUTORESEARCH_ROOT),
    "python": PYTHON,
})


{'claw_root': '/Users/wongdowling/Documents/autoresearch_harness/harness/claw-code', 'rust_dir': '/Users/wongdowling/Documents/autoresearch_harness/harness/claw-code/rust', 'autoresearch_root': '/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos', 'python': '/Users/wongdowling/opt/anaconda3/bin/python'}


In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────
OLLAMA_HOST  = "http://localhost:11434"
OLLAMA_MODEL = "qwen2.5-coder:7b"   # must be pulled: ollama pull qwen2.5-coder:7b

# The Rust claw binary reaches Ollama via the OpenAI-compat path:
#   OPENAI_BASE_URL  → Ollama /v1 endpoint
#   OPENAI_API_KEY   → any non-empty string (Ollama ignores it)
CLAW_OLLAMA_ENV = {
    "OPENAI_BASE_URL": f"{OLLAMA_HOST}/v1",
    "OPENAI_API_KEY": "local",
    # OPENAI_BASE_URL explicitly forces OpenAI-compat routing for non-claude/grok models.
    # Clearing the Anthropic env vars avoids accidental env-based provider selection.
    "ANTHROPIC_API_KEY": "",
    "ANTHROPIC_AUTH_TOKEN": "",
    # Suppress ANSI spinner so pipe-captured output is clean
    "NO_COLOR": "1",
    "CI": "1",
}

API_SERVER_PORT = 17332   # separate port so it doesn't conflict with other sessions
API_SERVER_HOST = "127.0.0.1"
API_BASE = f"http://{API_SERVER_HOST}:{API_SERVER_PORT}"


## 1 · Verify prerequisites

In [3]:
# Check cargo is available
cargo_result = subprocess.run(["cargo", "--version"], capture_output=True, text=True)
cargo_ok = cargo_result.returncode == 0
cargo_version = cargo_result.stdout.strip()

# Check Ollama is up
ollama_ok = ollama_ready(OLLAMA_HOST)

# Check the Ollama model is pulled
model_ok = False
if ollama_ok:
    try:
        with urllib.request.urlopen(f"{OLLAMA_HOST}/api/tags", timeout=3) as r:
            tags = json.loads(r.read())
        model_ok = any(OLLAMA_MODEL in m.get("name", "") for m in tags.get("models", []))
    except Exception:
        pass

print(f"cargo      : {'✓  ' + cargo_version if cargo_ok else '✗  not found – install Rust toolchain'}")
print(f"ollama     : {'✓  reachable at ' + OLLAMA_HOST if ollama_ok else '✗  not reachable – run: ollama serve'}")
print(f"model      : {'✓  ' + OLLAMA_MODEL + ' is pulled' if model_ok else '✗  not pulled – run: ollama pull ' + OLLAMA_MODEL}")

if not cargo_ok:
    raise RuntimeError("cargo not found. Install from https://rustup.rs")
if not ollama_ok:
    raise RuntimeError(f"Ollama is not reachable at {OLLAMA_HOST}. Run: ollama serve")
if not model_ok:
    raise RuntimeError(f"Model {OLLAMA_MODEL!r} is not pulled. Run: ollama pull {OLLAMA_MODEL}")


cargo      : ✓  cargo 1.94.1 (Homebrew)
ollama     : ✓  reachable at http://localhost:11434
model      : ✓  qwen2.5-coder:7b is pulled


## 2 · Build `claw` (skipped if binary already exists)

In [4]:
binary_exists = CLAW_BIN.exists()
binary_mtime = CLAW_BIN.stat().st_mtime if binary_exists else 0.0
source_mtime = latest_rust_source_mtime(RUST_DIR)
needs_rebuild = (not binary_exists) or (source_mtime > binary_mtime)

if needs_rebuild:
    print(f"Building Rust workspace in {RUST_DIR}  (this takes ~2–5 min on first build) …")
    run_cmd(
        ["cargo", "build", "--workspace"],
        cwd=RUST_DIR,
        timeout=600,   # 10-minute cap for first-time compilation
    )
    assert CLAW_BIN.exists(), f"Expected binary at {CLAW_BIN} but it was not produced"
else:
    print(f"Binary already built and up to date: {CLAW_BIN}")
    print("Skipping cargo build because no Rust sources are newer than the binary.")

print(f"\n✓  claw binary: {CLAW_BIN}  ({CLAW_BIN.stat().st_size // 1024} KB)")


Binary already built and up to date: /Users/wongdowling/Documents/autoresearch_harness/harness/claw-code/rust/target/debug/claw
Skipping cargo build because no Rust sources are newer than the binary.

✓  claw binary: /Users/wongdowling/Documents/autoresearch_harness/harness/claw-code/rust/target/debug/claw  (39011 KB)


## 3 · Confirm `claw --help` runs cleanly

In [5]:
help_result = run_cmd([CLAW_BIN, "--help"], cwd=RUST_DIR, check=False, timeout=10)
assert help_result.returncode == 0, f"--help exited with {help_result.returncode}"
assert "claw" in (help_result.stdout + help_result.stderr).lower(), "Expected 'claw' in help output"
print("✓  --help exit code:", help_result.returncode)


$ /Users/wongdowling/Documents/autoresearch_harness/harness/claw-code/rust/target/debug/claw --help
claw v0.1.0

Usage:
  claw [--model MODEL] [--allowedTools TOOL[,TOOL...]]
      Start the interactive REPL
  claw [--model MODEL] [--output-format text|json] prompt TEXT
      Send one prompt and exit
  claw [--model MODEL] [--output-format text|json] TEXT
      Shorthand non-interactive prompt mode
  claw --resume [SESSION.jsonl|session-id|latest] [/status] [/compact] [...]
      Inspect or maintain a saved session without entering the REPL
  claw help
      Alias for --help
  claw version
      Alias for --version
  claw status
      Show the current local workspace status snapshot
  claw sandbox
      Show the current sandbox isolation snapshot
  claw doctor
      Diagnose local auth, config, workspace, and sandbox health
  claw dump-manifests
  claw bootstrap-plan
  claw agents
  claw mcp
  claw skills
  claw system-prompt [--cwd PATH] [--date YYYY-MM-DD]
  claw login
  claw logout


## 4 · Ping Ollama through `claw` (one-shot prompt)

`claw` selects the OpenAI-compatible provider when `OPENAI_BASE_URL` is set and
the model name does not start with `claude` or `grok`.  Pointing it at
`http://localhost:11434/v1` routes every request through Ollama's
`/v1/chat/completions` endpoint.


In [6]:
ping_result = run_cmd(
    [CLAW_BIN, "--model", OLLAMA_MODEL, "prompt", "Reply with exactly one word: ready"],
    cwd=RUST_DIR,
    env=CLAW_OLLAMA_ENV,
    check=False,
    timeout=300,
)
combined = (ping_result.stdout + ping_result.stderr).strip().lower()
print("exit code :", ping_result.returncode)
print("response  :", combined[:200])

# A successful ping contains the word "ready" somewhere in the output
assert "ready" in combined, (
    f"Expected 'ready' in claw response but got: {combined[:300]!r}\n"
    "Check that OPENAI_BASE_URL is reachable and the model is loaded."
)
print("\n✓  claw reached Ollama and got a coherent response")


$ /Users/wongdowling/Documents/autoresearch_harness/harness/claw-code/rust/target/debug/claw --model qwen2.5-coder:7b prompt 'Reply with exactly one word: ready'
7⠋ 🦀 Thinking...8ready✔ ✨ Done


exit code : 0
response  : 7⠋ 🦀 thinking...8ready✔ ✨ done


✓  claw reached Ollama and got a coherent response


## 5 · JSON output mode

`--output-format json` makes `claw` emit a machine-readable result object.
This is how the Rust harness integrates with scripts and notebooks.


In [7]:
json_result = run_cmd(
    [
        CLAW_BIN,
        "--output-format", "json",
        "--model", OLLAMA_MODEL,
        "prompt", "What is 2 + 2? Reply with only the number.",
    ],
    cwd=RUST_DIR,
    env=CLAW_OLLAMA_ENV,
    check=False,
    timeout=300,
)
print("exit code:", json_result.returncode)

# claw --output-format json writes one JSON object per line; grab the last one
json_lines = [l for l in json_result.stdout.splitlines() if l.strip().startswith("{")]
assert json_lines, f"No JSON lines in output:\n{json_result.stdout[:500]}"

parsed = json.loads(json_lines[-1])
print("parsed keys:", list(parsed.keys()))
# The result should contain a 'result' or 'content' key with the answer
answer_text = str(parsed.get("result", parsed.get("content", parsed))).lower()
assert "4" in answer_text, f"Expected '4' in answer but got: {answer_text!r}"
print("\n✓  --output-format json works; answer contains '4'")
print(json.dumps(parsed, indent=2)[:600])


$ /Users/wongdowling/Documents/autoresearch_harness/harness/claw-code/rust/target/debug/claw --output-format json --model qwen2.5-coder:7b prompt 'What is 2 + 2? Reply with only the number.'
{"auto_compaction":null,"estimated_cost":"$0.0616","iterations":1,"message":"4","model":"qwen2.5-coder:7b","prompt_cache_events":[],"tool_results":[],"tool_uses":[],"usage":{"cache_creation_input_tokens":0,"cache_read_input_tokens":0,"input_tokens":4096,"output_tokens":2}}

exit code: 0
parsed keys: ['auto_compaction', 'estimated_cost', 'iterations', 'message', 'model', 'prompt_cache_events', 'tool_results', 'tool_uses', 'usage']

✓  --output-format json works; answer contains '4'
{
  "auto_compaction": null,
  "estimated_cost": "$0.0616",
  "iterations": 1,
  "message": "4",
  "model": "qwen2.5-coder:7b",
  "prompt_cache_events": [],
  "tool_results": [],
  "tool_uses": [],
  "usage": {
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "input_tokens": 4096,
    "output_to

## 6 · Start the Python autoresearch API server

The Python `api-server` subcommand exposes the full control plane as a local
REST API.  The Rust `claw` binary can then act as the **manager** by calling
this API via its bash tool — no Python imports needed on the manager side.


In [8]:
# Start the server as a background subprocess
api_proc = subprocess.Popen(
    [
        PYTHON, "-m", "src.main",
        "api-server",
        "--root", str(AUTORESEARCH_ROOT),
        "--port", str(API_SERVER_PORT),
        "--listen", API_SERVER_HOST,
        "--backend", "ollama",
        "--host", OLLAMA_HOST,
        "--model", OLLAMA_MODEL,
    ],
    cwd=str(CLAW_ROOT),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
    text=True,
)

# Confirm the server started (poll so we don't hang on a failed start)
if api_proc.poll() is not None:
    err = api_proc.stderr.read()
    raise RuntimeError(f"API server exited immediately. stderr:\n{err}")

try:
    health = wait_for_json(f"{API_BASE}/health", timeout=30)
except Exception as e:
    api_proc.terminate()
    err = api_proc.stderr.read()
    raise RuntimeError(f"API server did not respond to /health: {e}\nstderr:\n{err}") from e

print("✓  API server running on", API_BASE)
print("   /health response:", health)


✓  API server running on http://127.0.0.1:17332
   /health response: {'status': 'ok', 'root': '/Users/wongdowling/Documents/autoresearch_harness/nodes/autoresearch-macos'}


## 7 · Call the API server directly with `curl` / urllib

In [9]:
try:
    # GET /status
    status = http_get_json(f"{API_BASE}/status")
    print("GET /status:")
    print("  current_branch  :", status.get("state", {}).get("branch"))
    print("  best_bpb        :", status.get("state", {}).get("best_bpb"))
    print("  isolated_branch :", status.get("isolated_branch"))

    # GET /log  (parse the existing run.log if present)
    try:
        log_metrics = http_get_json(f"{API_BASE}/log")
        print("\nGET /log:")
        print("  val_bpb         :", log_metrics.get("val_bpb"))
        print("  success         :", log_metrics.get("success"))
    except Exception as log_err:
        print("\nGET /log skipped (no run.log yet):", log_err)

    print("\n✓  API server responds correctly to /status and /log")
finally:
    pass   # keep server up for the next cell


GET /status:
  current_branch  : autoresearch/notebook-apr08-0353
  best_bpb        : 0.999
  isolated_branch : True

GET /log:
  val_bpb         : 0.999
  success         : True

✓  API server responds correctly to /status and /log


## 8 · `claw` (Ollama) calls the API server via its bash tool

This is the key end-to-end test: the Rust harness uses its built-in bash tool
to call `curl` against the Python REST API.  This is how `claw` acts as the
manager in a real research session — it asks the LLM to plan an experiment,
then issues HTTP calls to the control plane to advance state.


In [10]:
try:
    manager_result = run_cmd(
        [
            CLAW_BIN,
            "--model", OLLAMA_MODEL,
            "--permission-mode", "danger-full-access",
            "--output-format", "json",
            "prompt",
            (
                f"Use the bash tool to run: curl -s {API_BASE}/health  "
                f"Then run: curl -s {API_BASE}/status  "
                "Return a JSON object with keys 'health' and 'branch' "
                "where 'branch' is the current_branch value from /status."
            ),
        ],
        cwd=str(RUST_DIR),
        env=CLAW_OLLAMA_ENV,
        check=False,
        timeout=300,
    )

    print("exit code:", manager_result.returncode)
    combined_output = manager_result.stdout + manager_result.stderr

    # Check the output contains expected API data
    health_seen = '"status"' in combined_output or "ok" in combined_output.lower()
    branch_seen = "autoresearch" in combined_output or "master" in combined_output or "main" in combined_output

    print("output excerpt:", combined_output[:600])
    print()

    if manager_result.returncode == 0 and health_seen:
        print("✓  claw (Ollama) successfully called the Python API server via bash tool")
        if branch_seen:
            print("✓  API /status branch info reached the LLM response")
    else:
        # Softer failure: flag it but don't crash the notebook —
        # the LLM may phrase its bash invocation differently across models.
        print("⚠  claw ran but output did not clearly confirm API round-trip.")
        print("   This can happen if the model chose not to use the bash tool.")
        print("   Re-run with a stronger model or check the trace output above.")
finally:
    # Always clean up the API server
    api_proc.terminate()
    try:
        api_proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        api_proc.kill()
    print("\nAPI server stopped.")


$ /Users/wongdowling/Documents/autoresearch_harness/harness/claw-code/rust/target/debug/claw --model qwen2.5-coder:7b --permission-mode danger-full-access --output-format json prompt 'Use the bash tool to run: curl -s http://127.0.0.1:17332/health  Then run: curl -s http://127.0.0.1:17332/status  Return a JSON object with keys '"'"'health'"'"' and '"'"'branch'"'"' where '"'"'branch'"'"' is the current_branch value from /status.'
{"auto_compaction":null,"estimated_cost":"$0.0704","iterations":1,"message":"{\n  \"name\": \"Bash\",\n  \"arguments\": {\n    \"commands\": [\n      \"curl -s http://127.0.0.1:17332/health\",\n      \"current_branch=$(curl -s http://127.0.0.1:17332/status | jq -r .current_branch)\",\n      \"echo \\\"{ \\\\\\\"health\\\\\\\": $(cat), \\\\\\\"branch\\\\\\\": \\\\\\\"$current_branch\\\\\\\\\" }\\\"\"\n    ]\n  },\n  \"includeStderr\": true\n}","model":"qwen2.5-coder:7b","prompt_cache_events":[],"tool_results":[],"tool_uses":[],"usage":{"cache_creation_input_toke

## Success Criteria

This notebook passes when:

| # | Check | How |
|---|---|---|
| 1 | `cargo` is on PATH and Ollama is reachable | Cell 3 |
| 2 | `cargo build --workspace` produces `rust/target/debug/claw` | Cell 4 |
| 3 | `claw --help` exits 0 | Cell 5 |
| 4 | `claw prompt "... ready"` via Ollama returns a response containing "ready" | Cell 6 |
| 5 | `claw --output-format json` emits parseable JSON with a numeric answer | Cell 7 |
| 6 | Python `api-server` starts, binds, and responds to `/health` | Cell 8 |
| 7 | urllib calls to `/status` and `/log` return structured JSON | Cell 9 |
| 8 | `claw` (Ollama, bash tool) calls the API server and surfaces the result | Cell 10 |

Checks 1–7 are hard assertions.  Check 8 is a soft check (⚠ warning rather than
error) because small quantized models may not reliably invoke bash tools on the
first attempt.  Re-run with `--model qwen2.5-coder:32b` or a larger model
for a higher success rate on the bash-tool round-trip.

### Architecture recap

```
claw (Rust + Ollama via OpenAI-compat)
  │  bash tool: curl http://localhost:17332/...
  ▼
Python api-server (src/api_server.py)
  │  control-plane functions
  ▼
autoresearch-macos node (results.tsv, state.json, train.py)
```
